# Notebook 02 — Corrective GraphRAG with LangGraph

Build a Corrective RAG (CRAG) agent that retrieves from a knowledge graph, grades relevance, and rewrites queries when needed.

The workflow combines LangGraph state transitions with structured knowledge-graph retrieval.

In [ ]:
import sys
from pathlib import Path
# Works from the repository root, experiment directory, or notebooks directory.
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((candidate for ancestor in _candidates
                     for candidate in (ancestor, ancestor / "experiments/graphrag")
                     if (candidate / "src/graphrag/config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter from this repository or its graphrag directory")
sys.path.insert(0, str(PROJECT_ROOT.parents[1]))
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")
load_dotenv(PROJECT_ROOT.parents[1] / ".env")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
import operator
from typing import Annotated, List, Optional, TypedDict, Literal

from langchain_core.messages import AnyMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from loguru import logger
import numpy as np

from graphrag import validate, get_neo4j_client, get_general_llm, get_strong_llm, get_embeddings
validate()

In [ ]:
client = get_neo4j_client()
if not client.verify_connectivity():
    client.close()
    raise ConnectionError("Neo4j is unavailable. Check the configured URI and credentials.")

node_count = client.get_node_count()
if node_count == 0:
    print("WARNING: Graph is empty! Run `make init-and-seed` or the ETL notebook first to populate it.")
else:
    print(f"Graph has {node_count} nodes — ready to go!")

## 1. Agent State

The state is a 'conveyor belt' — each node reads what it needs and writes its updates.

In [ ]:
from graphrag.agent import GraphRAGState
# messages accumulate; documents are replaced for each retrieval attempt.
print(GraphRAGState.__annotations__)

## 2. Prompts & Schemas

In [ ]:
# --- Load all prompts from central config ---
from graphrag.prompts import (
    ROUTER_SYSTEM_PROMPT,
    GRADER_SYSTEM_PROMPT,
    GRADER_HUMAN_PROMPT,
    REWRITER_SYSTEM_PROMPT,
    REWRITER_HUMAN_PROMPT,
    GENERATOR_SYSTEM_PROMPT,
    GENERATOR_HUMAN_PROMPT,
    GENERAL_MEDICAL_PROMPT,
    CHITCHAT_RESPONSE,
)

print("All prompts loaded from graphrag/prompts.py")

from graphrag.agent import RouteDecision, GradeDecision

## 3. Entity Resolution & Graph Retriever

These two classes power the retrieval pipeline: resolve user terms to graph entities, then traverse the graph with Cypher.

In [ ]:
from graphrag.retrieval import EntityResolver, ResolvedEntity, ExtractedQueryEntities
# Exact names/aliases, scored fuzzy matches, then label-aware semantic matches.

In [ ]:
from graphrag.retrieval import GraphRetriever
from graphrag import get_config
_params = get_config()
RETRIEVAL_MAX_HOPS = _params.get("retrieval", {}).get("max_hops", 2)
RETRIEVAL_TOP_K = _params.get("retrieval", {}).get("top_k", 5)
AGENT_MAX_RETRIES = _params.get("agent", {}).get("max_retries", 2)

In [ ]:
resolver = EntityResolver(client, embeddings=get_embeddings())
retriever = GraphRetriever(client, resolver)

# Quick test
docs = retriever.retrieve_for_question("What drugs interact with Warfari?", llm=get_general_llm())
print(f"Retrieved {len(docs)} documents:")
for doc in docs[:3]:
    print(f"  {doc.page_content[:100]}...")

## 4. Multi-Hop Queries — The Graph Advantage

This is where knowledge graphs shine over vector search. Let's see multi-hop traversal in action.

In [ ]:
# 1-HOP: Direct interactions
print("=== 1-HOP: Direct interactions with Warfarin ===")
results = client.query("""
    MATCH (d:Drug {name: 'Warfarin'})-[r:INTERACTS_WITH]-(other:Drug)
    RETURN other.name AS drug, r.severity AS severity
""")
for r in results:
    print(f"  {r['drug']} (severity: {r['severity']})")

# 2-HOP: What drug classes do Warfarin's interacting drugs belong to?
print("\n=== 2-HOP: Drug classes of drugs that interact with Warfarin ===")
results = client.query("""
    MATCH (d:Drug {name: 'Warfarin'})-[:INTERACTS_WITH]-(other:Drug)-[:BELONGS_TO_CLASS]->(dc:DrugClass)
    RETURN other.name AS drug, dc.name AS drug_class
""")
for r in results:
    print(f"  {r['drug']} belongs to class: {r['drug_class']}")

# CROSS-TYPE: Drugs sharing side effects with Warfarin
print("\n=== CROSS-TYPE: Drugs sharing side effects with Warfarin ===")
results = client.query("""
    MATCH (d:Drug {name: 'Warfarin'})-[:CAUSES_SIDE_EFFECT]->(se:SideEffect)<-[:CAUSES_SIDE_EFFECT]-(other:Drug)
    WHERE d <> other
    RETURN se.name AS side_effect, collect(other.name) AS other_drugs
""")
for r in results:
    print(f"  Shared side effect '{r['side_effect']}' with: {r['other_drugs']}")

## 5. CRAG Agent — All Nodes

The Corrective RAG pattern: Router -> Retrieve -> Grade -> (Rewrite loop) -> Generate.

We define all 6 node functions in one cell.

In [ ]:
from graphrag.agent import build_workflow
# Router -> retrieve -> grade -> generate, or rewrite with fresh context.
# Retry exhaustion goes to an explicit insufficient-context response.
# Entity extraction uses the strong model; routing/grading/generation use general.

## 6. Conditional Edges & LangGraph Assembly

In [ ]:
app = build_workflow(retriever, max_retries=AGENT_MAX_RETRIES)
print("Workflow compiled")

In [ ]:
from IPython.display import Markdown, display
# Mermaid source renders without calling a remote diagram service.
display(Markdown("```mermaid\n" + app.get_graph().draw_mermaid() + "\n```"))

## 7. Test the Agent

Run queries through the full CRAG pipeline.

In [ ]:
from graphrag.agent import run_query

# Test 1: Drug interaction
print("=" * 60)
print("Q: Does Warfarin interact with Aspirin?")
print("=" * 60)
answer = run_query(app, "Does Warfarin interact with Aspirin?")
print(f"\nAnswer:\n{answer}")

In [ ]:
# Test 2: Drug class query
print("=" * 60)
print("Q: What drugs are in the NSAID class?")
print("=" * 60)
answer = run_query(app, "What drugs are in the NSAID class?")
print(f"\nAnswer:\n{answer}")

In [ ]:
# Test 3: Non-domain question
print("=" * 60)
print("Q: Hello, how are you?")
print("=" * 60)
answer = run_query(app, "Hello, how are you?")
print(f"\nAnswer:\n{answer}")

## 8. Graph RAG vs Vector RAG

The critical comparison: when does each approach win?

In [ ]:
from graphrag.retrieval import VectorRetriever

import json
from pathlib import Path

# Load text passages
with open(PROJECT_ROOT / "data" / "raw" / "drug_interactions.json") as f:
    raw = json.load(f)
passages = [p["text"] for p in raw.get("text_passages", [])]

# Build vector index
vec_retriever = VectorRetriever(get_embeddings())
vec_retriever.index_passages(passages)
print(f"Vector index: {len(passages)} passages indexed")

comparison_questions = [
    "Does Warfarin interact with Aspirin?",
    "What drugs are contraindicated for patients with bleeding disorders?",
    "Which drugs share side effects with Metformin?",
]

for question in comparison_questions:
    print("=" * 70)
    print(f"Q: {question}")
    print("-" * 70)
    
    # Graph RAG
    graph_docs = retriever.retrieve_for_question(question, llm=get_general_llm())
    print(f"\nGraph RAG ({len(graph_docs)} results):")
    for doc in graph_docs[:3]:
        print(f"  {doc.page_content[:120]}...")
    
    # Vector RAG
    vec_docs = vec_retriever.retrieve(question, top_k=3)
    print(f"\nVector RAG ({len(vec_docs)} results):")
    for doc in vec_docs[:3]:
        score = doc.metadata.get('score', 'N/A')
        print(f"  [score: {score}] {doc.page_content[:120]}...")
    print()

### Analysis

| Aspect | Vector RAG | Graph RAG |
|---|---|---|
| Retrieval | Semantic similarity over text chunks | Structured Cypher over typed entities |
| Relationships | Implicit (buried in text) | Explicit typed edges |
| Multi-hop | Not possible | Native traversal |
| Best for | Broad semantic queries | Precise relational queries |
| Weakness | Misses structural relationships | Requires schema design |

**Key takeaway**: Graph RAG excels at relational queries ('what interacts with X?', 'what drugs share side effects?'). Vector RAG is better for broad semantic similarity. The ideal system combines both.

## Summary

| Concept | Implementation |
|---|---|
| GraphRAGState | TypedDict state with annotated message aggregation |
| CRAG Pattern | Router -> Retrieve -> Grade -> (Rewrite/Retry) -> Generate |
| Entity Resolution | 3-tier matching bridges user language to graph entities |
| Graph Retriever | Single-hop, multi-hop, and drug-interaction Cypher queries |
| Conditional Edges | LangGraph routes based on grading results |
| Graph vs Vector | Graph retrieval is better suited to explicit relational queries; vector retrieval remains useful for semantic similarity |

In [ ]:
client.close()
print("Done! All resources released.")